# Taiwan-Tongues ASR API (台語/國語/客語語音辨識)

本 Notebook 部署 Taiwan-Tongues 語音辨識 API，支援台語、國語、客語、英語。

**使用方式：**
1. 確認 Runtime → Change runtime type → T4 GPU
2. 依序執行所有 Cell
3. 最後會輸出 ngrok 公開 URL，複製到你的 `.env` 中

**注意：** 需要免費的 ngrok authtoken，到 https://ngrok.com 註冊取得

In [ ]:
# Cell 1: 安裝依賴
!pip install -q faster-whisper ctranslate2 flask pyngrok

In [ ]:
# Cell 2: 載入 Taiwan-Tongues ASR 模型 (支援台語/國語/客語/英語)
from faster_whisper import WhisperModel

# 直接從 Hugging Face 載入 v2.0 CTranslate2 模型
# GPU + float16 加速推論
model = WhisperModel(
    'adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0',
    device='cuda',
    compute_type='float16',
)
print('✅ Taiwan-Tongues ASR v2.0 模型載入完成 (GPU)')
print('支援語言: 國語(zh), 台語(nan), 客語(hak), 英語(en), 印尼語(id)')

In [ ]:
# Cell 3: 建立 Flask API 服務
import tempfile
from flask import Flask, request, jsonify
import threading
import os

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'Taiwan-Tongues-ASR-CE-v2.0'})

@app.route('/transcribe', methods=['POST'])
def transcribe():
    if 'audio' not in request.files:
        return jsonify({'success': False, 'message': '請上傳音檔'}), 400

    audio_file = request.files['audio']

    # 儲存暫存檔
    with tempfile.NamedTemporaryFile(suffix='.webm', delete=False) as tmp:
        audio_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        # 不指定語言，讓模型自動偵測（支援 zh/nan/hak/en/id）
        segments, info = model.transcribe(
            tmp_path,
            language=None,
            beam_size=5,
            best_of=3,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=500),
        )

        text = ' '.join([seg.text.strip() for seg in segments])
        detected_lang = info.language

        return jsonify({
            'success': True,
            'text': text,
            'language': detected_lang,
            'language_probability': round(info.language_probability, 3)
        })
    except Exception as e:
        return jsonify({'success': False, 'message': str(e)}), 500
    finally:
        os.unlink(tmp_path)

print('✅ Flask API 已定義')

In [ ]:
# Cell 5: 設定 ngrok 並啟動服務
# 請替換為你的 ngrok authtoken
# 免費註冊: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTHTOKEN = 'YOUR_NGROK_AUTHTOKEN_HERE'  # ← 替換這裡

from pyngrok import ngrok

# 設定 authtoken
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 開啟 ngrok tunnel
public_url = ngrok.connect(5000)
print('=' * 60)
print(f'🎉 ASR API 已上線！')
print(f'📡 公開 URL: {public_url}')
print(f'')
print(f'請將以下設定加到你的 backend/.env：')
print(f'TAIWAN_ASR_URL={public_url}/transcribe')
print('=' * 60)

# 啟動 Flask（在背景執行）
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000)).start()
print('\nFlask 服務已啟動，此 Cell 請保持執行中...')

In [ ]:
# Cell 6 (可選): 測試 API
import requests

# 健康檢查
resp = requests.get('http://localhost:5000/health')
print('Health:', resp.json())

# 如果有測試音檔可以在這裡測試
# with open('test.wav', 'rb') as f:
#     resp = requests.post('http://localhost:5000/transcribe', files={'audio': f})
#     print('Transcribe:', resp.json())

In [ ]:
# Cell 7: 保持 Notebook 不斷線（免費 Colab 約 90 分鐘會斷）
import time
while True:
    time.sleep(60)
    print('.', end='', flush=True)